In [1]:


import os
import sys
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, parent_dir)

import pickle
import numpy as np
import math
from Camera import Camera
from Frame import Frame
import matplotlib.pyplot as plt
import Utils
%matplotlib qt



from Frame import Frame

path = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov7_2024_11_12_darkan/'

# D:\Documents\data_for_gs\bee\images

path = 'D:/Documents/data_for_gs/mov19_2022_03_03/'

path = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov19_2022_03_03/'
path = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov1_2023_08_09_60ms/'

frames = list(range(1450,1700,1))
image_name= []
for frame in frames:
    image_name += [f'P{frame}CAM{cam + 1}' for cam in range(4)]

frames = {f'{im_name}.jpg':Frame(path,im_name,idx, camera_path = path,delta_xy = 80, use_mask = False) for idx,im_name in enumerate(image_name)}


KeyboardInterrupt: 

In [29]:
frames_dict = {}
frames_numbers = range(1450,1700,1)
for frame_number in frames_numbers:
    cams,base_images,frame_mov  = {},{},{}
    for idx in range(4):
        frame_name = f'P{frame_number}CAM{idx+1}.jpg'
        frame = frames[frame_name]
        base_images[frame.image_id] = frame.generate_base_image()
        frame.save_croped_images()
        cams[frame.camera_number] = frame.cams_for_gs()
        if idx == 0:
            rot_z = frames[frame_name].rotation_matrix_from_vectors(frames[frame_name].R[2,:], [0,0,1])
        cams[frame.camera_number]['ew_to_lab'] = rot_z
        cams[frame.camera_number]['bounding_box'] = frame.bounding_box
        frame_mov['image_path'] = f'{frame.path}/images/'
        frame_mov['real_frames'] = frames_numbers

    frames_dict[frame_number] = [base_images,cams,[],frame_mov]

Utils.pickle_file(frames_dict,f'{frame.path}/frames_model_seminar.pkl' )

In [21]:
frames_dict[int(key.split('_')[3])][2]

[]

In [13]:
for key,angles in zip(keys,data_txt):
    if int(key.split('_')[3]) in frames_dict:
        wajj =2
    else:
        print(2)



In [23]:
import os
import sys
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, parent_dir)

import pickle
import numpy as np
import math
from Camera import Camera
from Frame import Frame
import matplotlib.pyplot as plt
import Utils
%matplotlib qt



from Frame import Frame
frames_dict = {}


import numpy as np
path_frames_mov_eval = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/evaluation'
data_txt = np.loadtxt(f'{path_frames_mov_eval}/output_formatted.txt')
movs = np.unique(data_txt[:,-1])


for movs_num in movs:
    movs_num = movs_num.astype(int)
    path = f'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov{movs_num}_2023_08_09_60ms/'

    frames_numbers = data_txt[data_txt[:,10] == movs_num,9].astype(int)
    image_name= []
    for frame in frames_numbers:
        image_name += [f'P{frame}CAM{cam + 1}' for cam in range(4)]

    frames = {f'{im_name}.jpg':Frame(path,im_name,idx, camera_path = path,delta_xy = 80, use_mask = False) for idx,im_name in enumerate(image_name)}

    
    for frame_number in frames_numbers:
        if frame_number in frames_dict:
            continue
        cams,base_images,frame_mov  = {},{},{}
        for idx in range(4):
            frame_original_name = f'P{frame_number}CAM{idx+1}.jpg'
            frame = frames[frame_original_name]
            base_images[frame.image_id] = frame.generate_base_image()
            frame_mov['real_frames'] = frames_numbers
            frame_mov['mov_name'] = movs_num
            frame_mov['image_path'] = f'{frame.path}/images/'

            frame.save_croped_images(image_name = f'mov{movs_num}_P{frame_number}CAM{idx+1}.jpg')
            cams[frame.camera_number] = frame.cams_for_gs()
            if idx == 0:
                rot_z = frames[frame_original_name].rotation_matrix_from_vectors(frames[frame_original_name].R[2,:], [0,0,1])
            cams[frame.camera_number]['ew_to_lab'] = rot_z
            cams[frame.camera_number]['bounding_box'] = frame.bounding_box
            
        frames_dict[frame_number] = [base_images,cams,[],frame_mov]

Utils.pickle_file(frames_dict,f'{path_frames_mov_eval}/frames_model_evaluation.pkl' )


In [25]:
frames_dict[1620][3]

{'real_frames': array([1620,  977,  890,  983,  960,  834,  757,  483,  839, 1104, 1692,
         863,  854]),
 'mov_name': 128,
 'image_path': 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov128_2023_08_09_60ms//images/'}

In [27]:
keys = [f'mov_{mov.astype(int)}_frame_{frame.astype(int)}' for frame,mov in zip(frames,mov)]


AttributeError: 'str' object has no attribute 'astype'

In [35]:
import numpy as np
path_frames_mov_eval = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/evaluation/output_formatted.txt'
path_to_save = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/evaluation'
data_txt = np.loadtxt(path_frames_mov_eval)
movs = np.unique(data_txt[:,-1])

frames = data_txt[:,-2]
mov = data_txt[:,-1]


angle_dict = {}
keys = [f'mov_{mov.astype(int)}_frame_{frame.astype(int)}' for frame,mov in zip(frames,mov)]
for key,angles in zip(keys,data_txt):
    if int(key.split('_')[1]) == frames_dict[int(key.split('_')[3])][3]['mov_name']:
        angle_dict[key] = {'body_angles':[float(angles[1]),-float(angles[0]),float(angles[2])]
                            ,'left_wing_angles': [-90+float(angles[6]),-float(angles[8]),float(angles[7])]
                            ,'right_wing_angles': [90-float(angles[3]),-float(angles[5]),-float(angles[4])],
                                        'right_wing_angle_joint1' : 0.0,
                                        'left_wing_angle_joint1' : -0.0,
                                        'right_wing_angle_joint2' : 0.0,
                                        'left_wing_angle_joint2' : -0.0,

                                        'right_wing_twist_joint1' : 0.0,
                                        'left_wing_twist_joint1' : -0.0,
                                        'right_wing_twist_joint2' : 0.0,
                                        'left_wing_twist_joint2' : -0.0}

Utils.pickle_file(angle_dict,f'{path_to_save}/nominal_initial_angles.pkl' )

In [2]:

import plotly.graph_objects as go

frames['P1400CAM1.jpg'].R[:,2]

rot_z = frames['P1400CAM1.jpg'].rotation_matrix_from_vectors(frames['P1400CAM1.jpg'].R[2,:], [0,0,1])

rot_z @ frames['P1400CAM1.jpg'].R 


center_cam = np.hstack([rot_z @ frame.X0 for frame in list(frames.values())[0:4]]).T

import Plotters
fig = go.Figure()
Plotters.scatter3d(fig,center_cam,'red',10,'centers')

KeyError: 'P1400CAM1.jpg'

In [25]:
frame.path

'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov7_2024_11_12_darkan/'

In [5]:
frames['P370CAM1.jpg'].bounding_box

array([538, 713, 698, 873])

In [50]:
frames_dict[1400][1][0]['ew_to_lab']

array([[ 0.99979039,  0.01822399, -0.00933071],
       [ 0.01822399, -0.58443755,  0.81123402],
       [ 0.00933071, -0.81123402, -0.58464716]])